# Exact matcher: unit-test development notebook

This notebook lives in **`docs/`** and is meant for *interactive* development of unit tests.

It uses the project's canonical entry point for matching:

- `TreePathMatcher(method="exact")` from `src/path_matcher/matcher.py`

and the project's sampler/planting conventions:

- trees are `igraph.Graph` with vertex attrs `"label"` and `"is_planted"`.

The helper code is in `tests/dev_utils.py` (samplers + planting + plotting).


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)


In [ ]:
# Imports + repo path

import os, sys
import numpy as np
import pandas as pd
import igraph as ig

# Make local source packages and test helpers importable when running from docs/.
REPO_ROOT = find_repo_root() if "find_repo_root" in globals() else Path(os.getcwd()).resolve().parent
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

from tests.dev_utils import (
    case_root_leaf_inclusion,
    case_no_matches,
    case_weighted_vs_unweighted_blockswap,
    summarize_case,
    plot_case,
)


## 1) Root/leaf inclusion sweep (16 combinations)

We generate **two small narrow trees** and plant the same length-`seg_len` match segment,
but we shift the segment so it may include/exclude the **root** and/or **leaf** in **each**
tree.

The result object also reports the 4 booleans:

- `truth_includes_root_G`, `truth_includes_leaf_G`, `truth_includes_root_H`, `truth_includes_leaf_H`


In [ ]:
# Root/leaf inclusion sweep

seed = 0
seg_len = 6

rows = []
results = {}

for include_root_G in [True, False]:
    for include_leaf_G in [True, False]:
        for include_root_H in [True, False]:
            for include_leaf_H in [True, False]:
                res = case_root_leaf_inclusion(
                    seed=seed,
                    seg_len=seg_len,
                    include_root_G=include_root_G,
                    include_leaf_G=include_leaf_G,
                    include_root_H=include_root_H,
                    include_leaf_H=include_leaf_H,
                    # sampler knobs (edit freely)
                    stub_prob=0.75,
                    stubs_per_spine_vertex=1,
                    stub_chain_length=1,
                    noise_k=2,
                    weight_mode="increasing",
                )
                results[(include_root_G, include_leaf_G, include_root_H, include_leaf_H)] = res
                rows.append(summarize_case(res))

df = pd.DataFrame(rows).sort_values(["rootG","leafG","rootH","leafH"], ascending=[False,False,False,False])
df


Pick a specific combination and inspect the trees + matches visually.

In [ ]:
# Visualize one case from the sweep

key = (False, True, True, False)  # (rootG, leafG, rootH, leafH) -- edit me
res = results[key]

print(res.name)
print("truth_pairs:", res.truth_pairs)
print("found_pos_pairs:", res.found_pos_pairs)
print(f"truth_score={res.truth_score}  found_pos_score={res.found_pos_score}  found_score={res.found_score}")

plot_case(res)


In [ ]:
# Root should not 

# Visualize one case from the sweep

key = (False, True, False, False)  # (rootG, leafG, rootH, leafH) -- edit me
res = results[key]

print(res.name)
print("truth_pairs:", res.truth_pairs)
print("found_pos_pairs:", res.found_pos_pairs)
print(f"truth_score={res.truth_score}  found_pos_score={res.found_pos_score}  found_score={res.found_score}")

plot_case(res)


In [ ]:
# Visualize one case from the sweep

key = (True, True, True, True)  # (rootG, leafG, rootH, leafH) -- edit me
res = results[key]

print(res.name)
print("truth_pairs:", res.truth_pairs)
print("found_pos_pairs:", res.found_pos_pairs)
print(f"truth_score={res.truth_score}  found_pos_score={res.found_pos_score}  found_score={res.found_score}")

plot_case(res)


## 2) No matches

Two trees with disjoint alphabets ⇒ the positive-score match should be empty and the score should be 0.


In [ ]:
# No-matches case

res0 = case_no_matches(seed=123, spine_length_G=10, spine_length_H=10)
print(res0.name)
print("found_pos_pairs:", res0.found_pos_pairs)
print(f"found_pos_score={res0.found_pos_score}  found_score={res0.found_score}")

plot_case(res0, title="No matches (disjoint alphabets)")


## 3) Weighted vs unweighted identity (path block-swap)

We build two **path trees**:

- `G = A0 A1 A2 B0 B1 B2 B3`
- `H = B0 B1 B2 B3 A0 A1 A2`

so any order-respecting alignment must choose between matching block **A** or block **B** (classic LCS tradeoff).

- Unweighted identity (`1[a=b]`) prefers **B** (length 4)
- Weighted identity (`weight[a]·1[a=b]`) prefers **A** if its weights are large enough


In [ ]:
cmp = case_weighted_vs_unweighted_blockswap(seed=7)

print("=== Weighted identity ===")
print("truth_pairs:", cmp.weighted.truth_pairs)
print("found_pos_pairs:", cmp.weighted.found_pos_pairs)
print(f"truth_score={cmp.weighted.truth_score}  found_pos_score={cmp.weighted.found_pos_score}  found_score={cmp.weighted.found_score}")

plot_case(cmp.weighted, title="Weighted identity: should pick A-block")

print("\n=== Unweighted identity ===")
print("truth_pairs:", cmp.unweighted.truth_pairs)
print("found_pos_pairs:", cmp.unweighted.found_pos_pairs)
print(f"truth_score={cmp.unweighted.truth_score}  found_pos_score={cmp.unweighted.found_pos_score}  found_score={cmp.unweighted.found_score}")

plot_case(cmp.unweighted, title="Unweighted identity: should pick B-block")
